# OKCupid Date-A-Scientist: Predicting Zodiac Signs from OKCupid Profiles

## The Problem
Zodiac compatibility is a big deal for a lot of OkCupid users. Analyzing the data in the previous notebook showed that 18.4% of profiles don't have a zodiac sign filled in at all. That's a pretty significant chunk of users who could be getting worse matches just because they skipped that one field (roughly 1 in 5 users!).

## The Goal
The idea is to use machine learning to take a guess at what someone's zodiac sign might be, based on other things we already know about them like their lifestyle, habits, and demographic info.  If we can predict it reasonably well, we can fill in those gaps and potentially improve match quality.

## The Methodology
This seems like a multi-class classification problem. We can use feature, such as drinking habits, smoking, education, age, etc to predict someone's zodiac sign. The model will be trained on profiles where the sign is already known, then we can use the model to predict the ones that are missing. There is no true science to zodiac signs given that they are based on birthdates, but it would be interesting to find if there is a correlation among any features.

---------------------------------------------------------------------------------------

In [9]:
# Libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import LabelEncoder, StandardScaler, OrdinalEncoder
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer

from plot_style import set_plot_style
set_plot_style()

# Load Data

In [10]:
df = pd.read_csv("data/profiles.csv")
print(f"Dataset shape: {df.shape}")
print(f"Dataset columns: {df.columns.tolist()}")

Dataset shape: (59946, 31)
Dataset columns: ['age', 'body_type', 'diet', 'drinks', 'drugs', 'education', 'essay0', 'essay1', 'essay2', 'essay3', 'essay4', 'essay5', 'essay6', 'essay7', 'essay8', 'essay9', 'ethnicity', 'height', 'income', 'job', 'last_online', 'location', 'offspring', 'orientation', 'pets', 'religion', 'sex', 'sign', 'smokes', 'speaks', 'status']


# Preprocess Data

In [11]:
df["sign"].value_counts(dropna=False)

sign
NaN                                              11056
gemini and it&rsquo;s fun to think about          1782
scorpio and it&rsquo;s fun to think about         1772
leo and it&rsquo;s fun to think about             1692
libra and it&rsquo;s fun to think about           1649
taurus and it&rsquo;s fun to think about          1640
cancer and it&rsquo;s fun to think about          1597
pisces and it&rsquo;s fun to think about          1592
sagittarius and it&rsquo;s fun to think about     1583
virgo and it&rsquo;s fun to think about           1574
aries and it&rsquo;s fun to think about           1573
aquarius and it&rsquo;s fun to think about        1503
virgo but it doesn&rsquo;t matter                 1497
leo but it doesn&rsquo;t matter                   1457
cancer but it doesn&rsquo;t matter                1454
gemini but it doesn&rsquo;t matter                1453
taurus but it doesn&rsquo;t matter                1450
aquarius but it doesn&rsquo;t matter              1408
libra

In [12]:
df["sign_cleaned"] =df["sign"].str.extract(r"([A-Za-z]+)")[0].str.lower()

print(f"Unique signs after cleaning: {df['sign_cleaned'].dropna().unique()}")
print(f"Rows with missing sign: {df['sign_cleaned'].isnull().sum()}")
print(f"Rows without missing sign: {df['sign_cleaned'].notnull().sum()}")

Unique signs after cleaning: <ArrowStringArray>
[     'gemini',      'cancer',      'pisces',    'aquarius',      'taurus',
       'virgo', 'sagittarius',         'leo',       'aries',       'libra',
     'scorpio',   'capricorn']
Length: 12, dtype: str
Rows with missing sign: 11056
Rows without missing sign: 48890


In [17]:
results = []

for col in ['drinks', 'smokes', 'drugs', 'body_type', 'education']:
    vc = df[col].value_counts().dropna().reset_index()
    vc.columns = [col, f'{col}_count']
    results.append(vc)

pd.concat(results, axis=1)

,drinks,drinks_count,smokes,smokes_count,drugs,drugs_count,body_type,body_type_count,education,education_count
0,socially,41780.0,no,43896.0,never,37724.0,average,14652.0,graduated from college/university,23959
1,rarely,5957.0,sometimes,3787.0,sometimes,7732.0,fit,12711.0,graduated from masters program,8961
2,often,5164.0,when drinking,3040.0,often,410.0,athletic,11819.0,working on college/university,5712
3,not at all,3267.0,yes,2231.0,NaN,NaN,thin,4711.0,working on masters program,1683
4,very often,471.0,trying to quit,1480.0,NaN,NaN,curvy,3924.0,graduated from two-year college,1531
5,desperately,322.0,NaN,NaN,NaN,NaN,a little extra,2629.0,graduated from high school,1428
6,NaN,NaN,NaN,NaN,NaN,NaN,skinny,1777.0,graduated from ph.d program,1272
7,NaN,NaN,NaN,NaN,NaN,NaN,full figured,1009.0,graduated from law school,1122
8,NaN,NaN,NaN,NaN,NaN,NaN,overweight,444.0,working on two-year college,1074
9,NaN,NaN,NaN,NaN,NaN,NaN,jacked,421.0,dropped out of college/university,995


In [18]:
# ordinal encoding - need to convert categorical data to numeric for ML models
# variable position will become the number
drinks_order = ["not at all", "rarely", "socially", "often", "very often", "desperately"]
smokes_order = ["no",  "trying to quit", "when drinking", "sometimes", "yes"]
body_type_order = ["average", "fit", "athletic", "thin", "curvy", "a little extra", "skinny", "overweight", "used up", "rather not say", "jacked", "full figured", "dad bod", "more body type info", "other"]

# anything not encoded on the lists above become NaNs.
enc = OrdinalEncoder(categories=[drinks_order, smokes_order, body_type_order],handle_unknown='use_encoded_value',unknown_value=np.nan)

# filtering and transoforming the data
# selected columns to encode
cols = ['drinks', 'smokes', 'body_type']
df[["drinks_encoded", "smokes_encoded", "body_type_encoded"]] = enc.fit_transform(df[cols]) # converts strings to numbers

In [ ]:
# education variables seem all over the place
df['education'].value_counts()

education
graduated from college/university    23959
graduated from masters program        8961
working on college/university         5712
working on masters program            1683
graduated from two-year college       1531
graduated from high school            1428
graduated from ph.d program           1272
graduated from law school             1122
working on two-year college           1074
dropped out of college/university      995
working on ph.d program                983
college/university                     801
graduated from space camp              657
dropped out of space camp              523
graduated from med school              446
working on space camp                  445
working on law school                  269
two-year college                       222
working on med school                  212
dropped out of two-year college        191
dropped out of masters program         140
masters program                        136
dropped out of ph.d program            127
d

In [24]:
# will only focus on the education level for simplicity

def grab_education_level(edu):
    """ Convert the education variable to a more standardized format. Conver to
    a numerical level based on the highest level of education completed. If the 
    valude indicates that the user dropped out, the function will return the level
    below the one they dropped out of because it is the highest level of education
    they completed. Values that are still in progross (e.g., "working on xxx") are
    taken at face value and assigned that level.
    
    Args:
        edu (str): The education variable as a string.
        
    Returns:
        float: The education level as a number, where higher numbers indicate higher 
                levels of education. NaN values are returned as NaN.
                    0   = dropped out of high school
                    1   = high school
                    2   = two-year college
                    2.5 = space camp
                    3   = college/university
                    4   = masters
                    5   = ph.d, law, or med
    """
    if pd.isna(edu):
        return np.nan
    
    edu = edu.lower()
    
    if "dropped out of" in edu:
            if any(k in edu for k in ["ph.d", "law", "med", "college", "masters", "space camp", "high school"]):
                if any(k in edu for k in ["ph.d", "law", "med"]):
                    return 4  # dropped out of ph.d, law, or med
                elif "masters" in edu:
                    return 3  # dropped out of masters
                elif "space camp" in edu:
                    return 2  # dropped out of space camp, highest completed is two-year level
                elif "college/university" in edu:
                    return 2.5  # dropped out of college/university, highest completed is space camp level
                elif "high school" in edu:
                    return 0  # dropped out of high school
                
    if any(k in edu for k in ["ph.d", "law", "med"]):
        return 5  # ph.d, law, or med
    elif "masters" in edu:
        return 4  # masters
    elif "college/university" in edu:
        return 3  # college/university
    elif "space camp" in edu:
        return 2.5  # space camp
    elif "two-year college" in edu:
        return 2  # two-year college
    elif "high school" in edu:
        return 1  # high school
    return np.nan  # for any other cases that don't fit the above patterns
            
df["education_encoded"] = df["education"].apply(grab_education_level)
df["education_encoded"].value_counts().sort_index()

df["education_encoded"]

0        3.0
1        2.5
2        4.0
3        3.0
4        3.0
        ... 
59941    3.0
59942    3.0
59943    4.0
59944    3.0
59945    4.0
Name: education_encoded, Length: 59946, dtype: float64

In [ ]:
# diet

df['diet'].value_counts()

diet
mostly anything        16585
anything                6183
strictly anything       5113
mostly vegetarian       3444
mostly other            1007
strictly vegetarian      875
vegetarian               667
strictly other           452
mostly vegan             338
other                    331
strictly vegan           228
vegan                    136
mostly kosher             86
mostly halal              48
strictly halal            18
strictly kosher           18
halal                     11
kosher                    11
Name: count, dtype: int64

In [26]:
# Will focus on how strict diet is and not the type of diet 
# religions diets (e.g., kosher, halal) are treated as very strict.

def get_diet_strictness(diet):
    """ Convert the diet variable to a more standardized format. Will focus on how
    strict the diet is and not the type of diet. Halal and kosher diets are treated 
    as very strict diets by nature.
    
    Args:
        diet (str): The diet variable as a string; Nan if missing
        
    Returns:
        float: The diet strictness level as a number, where higher numbers indicate stricter
                    diets. NaN values are returned as NaN.
                        1 = mostly (e.g., 'mostly vegan', 'mostly anything')
                        2 = no modifier (e.g., "vegan", "vegetarian")
                        3 = strict (e.g., "strictly vegan", "kosher", "halal")
    """
    if pd.isna(diet):
        return np.nan
    
    diet = diet.lower()
    
    if diet.startswith("strictly") or "halal" in diet or "kosher" in diet:
        return 3  # strict diet
    elif diet.startswith("mostly"):
        return 1  # mostly diet
    return 2 # no modifier

df["diet_strictness"] = df["diet"].apply(get_diet_strictness)
df["diet_strictness"].value_counts().sort_index()

df["diet_strictness"]

0        3.0
1        1.0
2        2.0
3        2.0
4        NaN
        ... 
59941    NaN
59942    1.0
59943    1.0
59944    1.0
59945    NaN
Name: diet_strictness, Length: 59946, dtype: float64

In [27]:
# check the distribution of age
df['age'].value_counts().sort_index()

age
18      309
19      611
20      953
21     1282
22     1934
23     2592
24     3242
25     3531
26     3724
27     3685
28     3583
29     3295
30     3149
31     2735
32     2587
33     2206
34     1902
35     1755
36     1583
37     1427
38     1330
39     1172
40     1030
41      980
42     1072
43      858
44      708
45      643
46      578
47      529
48      481
49      459
50      437
51      350
52      344
53      252
54      267
55      265
56      271
57      256
58      197
59      221
60      195
61      176
62      167
63      138
64      113
65      109
66      105
67       66
68       59
69       31
109       1
110       1
Name: count, dtype: int64

In [28]:
# let's remove weird ages (e.g., 110)
df["age"] = df["age"].where(df["age"] < 100, np.nan)

df["age"].describe()

count    59944.000000
mean        32.337715
std          9.442423
min         18.000000
25%         26.000000
50%         30.000000
75%         37.000000
max         69.000000
Name: age, dtype: float64

In [29]:
# check all unique sign values
df['sign'].value_counts()

sign
gemini and it&rsquo;s fun to think about         1782
scorpio and it&rsquo;s fun to think about        1772
leo and it&rsquo;s fun to think about            1692
libra and it&rsquo;s fun to think about          1649
taurus and it&rsquo;s fun to think about         1640
cancer and it&rsquo;s fun to think about         1597
pisces and it&rsquo;s fun to think about         1592
sagittarius and it&rsquo;s fun to think about    1583
virgo and it&rsquo;s fun to think about          1574
aries and it&rsquo;s fun to think about          1573
aquarius and it&rsquo;s fun to think about       1503
virgo but it doesn&rsquo;t matter                1497
leo but it doesn&rsquo;t matter                  1457
cancer but it doesn&rsquo;t matter               1454
gemini but it doesn&rsquo;t matter               1453
taurus but it doesn&rsquo;t matter               1450
aquarius but it doesn&rsquo;t matter             1408
libra but it doesn&rsquo;t matter                1408
capricorn and it&rsquo;